# PROTOCOL-ARENA — Training Colab

This notebook runs the **two-phase training recipe** for PROTOCOL-ARENA on a single A100:

1. **Phase A — SFT Bootstrap**: teach the base model the MCP/A2A frame grammar.
2. **Phase B — GRPO Under Drift**: shape the policy with the five-signal reward.

Total wall-clock on a free Colab A100: ~90 minutes.

In [ ]:
# ---- install ----
!pip -q install unsloth 'trl>=0.8' peft accelerate bitsandbytes datasets
!pip -q install openenv-core fastapi uvicorn pydantic
!git clone https://github.com/<your-user>/protocol-arena.git || true
%cd protocol-arena

In [ ]:
# ---- generate SFT rollouts ----
!python -m arena.training.sft_bootstrap --out data/sft_rollouts.jsonl --episodes 400

In [ ]:
# ---- Phase A: SFT on oracle rollouts ----
import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

BASE = 'Qwen/Qwen2.5-7B-Instruct'
model, tok = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=2048,
                                              dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.0,
                                         target_modules=['q_proj','k_proj','v_proj','o_proj',
                                                         'gate_proj','up_proj','down_proj'])

ds = load_dataset('json', data_files='data/sft_rollouts.jsonl', split='train')
def fmt(row):
    msgs = row['messages']
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}
ds = ds.map(fmt)

trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
                     args=SFTConfig(output_dir='outputs/sft', num_train_epochs=1,
                                    per_device_train_batch_size=2, gradient_accumulation_steps=8,
                                    learning_rate=2e-4, logging_steps=10, max_seq_length=2048))
trainer.train()
model.save_pretrained('outputs/sft')
tok.save_pretrained('outputs/sft')

In [ ]:
# ---- Phase B: GRPO under drift ----
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
from arena.tasks import ALL_TASKS
from arena.server.arena_env import ProtocolArenaEnvironment
from arena.models import OrchestratorAction

prompts = Dataset.from_list([{'prompt': t['spec'], 'task_id': tid}
                             for tid, t in ALL_TASKS.items()])

def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for p, c in zip(prompts, completions):
        try: decision = json.loads(c)
        except Exception: rewards.append(-0.5); continue
        tid = kwargs.get('task_id', list(ALL_TASKS.keys())[0])
        env = ProtocolArenaEnvironment()
        env.reset(task_id=tid, seed=0)
        try:
            act = OrchestratorAction(**{k:v for k,v in decision.items()
                                       if k in {'kind','rationale','mcp_call','a2a_call',
                                                'dag_delta','kg_op','final'}})
            obs = env.step(act)
            rewards.append(float(obs.reward))
        except Exception: rewards.append(-0.2)
    return rewards

cfg = GRPOConfig(output_dir='outputs/grpo', learning_rate=5e-6,
                 num_generations=8, max_steps=200,
                 per_device_train_batch_size=1, gradient_accumulation_steps=4, beta=0.04)
grpo = GRPOTrainer(model=model, tokenizer=tok, reward_funcs=[reward_fn], args=cfg,
                   train_dataset=prompts)
grpo.train()
model.save_pretrained('outputs/grpo')
tok.save_pretrained('outputs/grpo')

In [ ]:
# ---- Evaluation on three splits ----
from arena.eval.harness import run_eval
from arena.eval.baselines import rule_based_policy

report = run_eval(rule_based_policy)
print(report)